# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates Elasticsearch's **MultiQuery Retriever** workflow.

The learning logic stays the same as in the original lab:

1. load a small dataset,
2. split documents into passages,
3. embed and index the passages in Elasticsearch,
4. use **MultiQuery Retriever** to generate related search queries,
5. answer a question from the retrieved context.

This version is adapted for **Google Colab** and **OpenAI via Colab Secrets**, so you do not need any local setup.

## Install packages and import modules

This Colab-friendly version uses the current LangChain package split:

- `langchain-openai` for OpenAI models and embeddings
- `langchain-elasticsearch` for the vector store
- `langchain-classic` for `MultiQueryRetriever`
- `langchain-text-splitters` for document chunking

Run this cell once after opening the notebook.

In [34]:
import subprocess, sys, os

INSTALL_FLAG = "/tmp/lab_mqr_installed"

if not os.path.exists(INSTALL_FLAG):
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-qU",
        "langchain-core",
        "langchain-classic",
        "langchain-openai",
        "langchain-elasticsearch",
        "langchain-text-splitters",
        "tiktoken",
    ], check=True)
    open(INSTALL_FLAG, "w").close()
    print("✅ Pakete installiert. Runtime wird neu gestartet...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    os.remove(INSTALL_FLAG)
    print("✅ Pakete bereits installiert — weiter geht's!")


✅ Pakete installiert/aktualisiert. Runtime wird neu gestartet...


{'status': 'ok', 'restart': True}

## Connect to Elasticsearch and OpenAI

This notebook reads credentials from **Colab Secrets**:

- `ELASTIC_CLOUD_ID`
- `ELASTIC_API_KEY`
- `OPENAI_API_KEY`

We use `.strip()` because secrets sometimes accidentally contain a leading or trailing blank space.

In [ ]:
!pip install elasticsearch

In [2]:
from elasticsearch import Elasticsearch, helpers
client = Elasticsearch(
    "https://my-elasticsearch-project-ea1419.es.europe-west3.gcp.elastic.cloud:443",
    api_key="RDJ3clRwMEJBeEpzUnJ4VElUNGE6VUpHZXV4ZjdpNDE3TkdCeWtldEgtdw=="
)
# -----------------------------
# Define the index name and create index (if it doesn't exist)
# -----------------------------
index_name = "national-parks"
if not client.indices.exists(index=index_name):
    create_response = client.indices.create(index=index_name)
    print("Index created:", create_response)
else:
    print(f"Index '{index_name}' already exists.")
# -----------------------------
# Add or update mappings for the index
# -----------------------------
mappings = {
    "properties": {
        "text": {
            "type": "semantic_text"
        }
    }
}
mapping_response = client.indices.put_mapping(
    index=index_name,
    body=mappings
)
print("Mappings updated:", mapping_response)
# -----------------------------
# Sample documents to ingest
# -----------------------------
docs = [
    {
        "text": "Yellowstone National Park is one of the largest national parks in the United States. It ranges from the Wyoming to Montana and Idaho, and contains an area of 2,219,791 acres across three different states. Its most famous for hosting the geyser Old Faithful and is centered on the Yellowstone Caldera, the largest super volcano on the American continent. Yellowstone is host to hundreds of species of animal, many of which are endangered or threatened. Most notably, it contains free-ranging herds of bison and elk, alongside bears, cougars and wolves. The national park receives over 4.5 million visitors annually and is a UNESCO World Heritage Site."
    },
    {
        "text": "Yosemite National Park is a United States National Park, covering over 750,000 acres of land in California. A UNESCO World Heritage Site, the park is best known for its granite cliffs, waterfalls and giant sequoia trees. Yosemite hosts over four million visitors in most years, with a peak of five million visitors in 2016. The park is home to a diverse range of wildlife, including mule deer, black bears, and the endangered Sierra Nevada bighorn sheep. The park has 1,200 square miles of wilderness, and is a popular destination for rock climbers, with over 3,000 feet of vertical granite to climb. Its most famous and cliff is the El Capitan, a 3,000 feet monolith along its tallest face."
    },
    {
        "text": "Rocky Mountain National Park is one of the most popular national parks in the United States. It receives over 4.5 million visitors annually, and is known for its mountainous terrain, including Longs Peak, which is the highest peak in the park. The park is home to a variety of wildlife, including elk, mule deer, moose, and bighorn sheep. The park is also home to a variety of ecosystems, including montane, subalpine, and alpine tundra. The park is a popular destination for hiking, camping, and wildlife viewing, and is a UNESCO World Heritage Site."
    }
]
# -----------------------------
# Bulk ingest documents
# -----------------------------
ingestion_timeout=300 # Allow time for semantic ML model to load
bulk_response = helpers.bulk(
    client.options(request_timeout=ingestion_timeout),
    docs,
    index=index_name,
    refresh="wait_for" # Wait until indexed documents are visible for search before returning the response
)
print(bulk_response)
# -----------------------------
# Define semantic search query
# -----------------------------
retriever_object = {
    "standard": {
        "query": {
            "semantic": {
                "field": "text",
                "query": "Sierra Nevada"
            }
        }
    }
}
search_response = client.search(
    index=index_name,
    retriever=retriever_object,
)
print(search_response['hits']['hits'])

Index 'national-parks' already exists.
Mappings updated: {'acknowledged': True}
(3, [])
[{'_index': 'national-parks', '_id': 'bmxfTp0BAxJsRrxTPT_f', '_score': 0.7587098, '_source': {'text': 'Yosemite National Park is a United States national park covering over 750,000 acres of land in California. A UNESCO World Heritage Site, the park is best known for its granite cliffs, waterfalls, and giant sequoia trees. Yosemite hosts over four million visitors in most years, with a peak of five million visitors in 2016. The park is home to a diverse range of wildlife, including mule deer, black bears, and the endangered Sierra Nevada bighorn sheep. The park has 1,200 square miles of wilderness and is a popular destination for rock climbers. One of its most famous cliffs is El Capitan, a 3,000-foot monolith along its tallest face.'}}, {'_index': 'national-parks', '_id': 'u2xoTp0BAxJsRrxTTz-B', '_score': 0.7579224, '_source': {'text': 'Yosemite National Park is a United States National Park, coveri

In [3]:
from google.colab import userdata
from elasticsearch import Elasticsearch, helpers

client = Elasticsearch(
    "https://my-elasticsearch-project-ea1419.es.europe-west3.gcp.elastic.cloud:443",
    api_key=userdata.get("https://my-elasticsearch-project-ea1419.es.europe-west3.gcp.elastic.cloud:443").strip(),
)

index_name = "national-parks"

if not client.indices.exists(index=index_name):
    client.indices.create(index=index_name)

mappings = {
    "properties": {
        "text": {"type": "semantic_text"}
    }
}

client.indices.put_mapping(index=index_name, body=mappings)

docs = [
    {
        "_index": index_name,
        "_source": {
            "text": "Yellowstone National Park is one of the largest national parks in the United States. It ranges from Wyoming to Montana and Idaho, and contains an area of 2,219,791 acres across three different states. It is most famous for hosting the geyser Old Faithful and is centered on the Yellowstone Caldera, the largest supervolcano on the American continent. Yellowstone is home to hundreds of animal species, many of which are endangered or threatened. Most notably, it contains free-ranging herds of bison and elk, alongside bears, cougars, and wolves. The national park receives over 4.5 million visitors annually and is a UNESCO World Heritage Site."
        },
    },
    {
        "_index": index_name,
        "_source": {
            "text": "Yosemite National Park is a United States national park covering over 750,000 acres of land in California. A UNESCO World Heritage Site, the park is best known for its granite cliffs, waterfalls, and giant sequoia trees. Yosemite hosts over four million visitors in most years, with a peak of five million visitors in 2016. The park is home to a diverse range of wildlife, including mule deer, black bears, and the endangered Sierra Nevada bighorn sheep. The park has 1,200 square miles of wilderness and is a popular destination for rock climbers. One of its most famous cliffs is El Capitan, a 3,000-foot monolith along its tallest face."
        },
    },
    {
        "_index": index_name,
        "_source": {
            "text": "Rocky Mountain National Park is one of the most popular national parks in the United States. It receives over 4.5 million visitors annually and is known for its mountainous terrain, including Longs Peak, which is the highest peak in the park. The park is home to a variety of wildlife, including elk, mule deer, moose, and bighorn sheep. It also contains montane, subalpine, and alpine tundra ecosystems. The park is a popular destination for hiking, camping, and wildlife viewing."
        },
    },
]

helpers.bulk(
    client.options(request_timeout=300),
    docs,
    refresh="wait_for",
)

retriever_object = {
    "standard": {
        "query": {
            "semantic": {
                "field": "text",
                "query": "Sierra Nevada"
            }
        }
    }
}

search_response = client.search(
    index=index_name,
    retriever=retriever_object,
)

print(search_response["hits"]["hits"])

[{'_index': 'national-parks', '_id': 'vmxoTp0BAxJsRrxTdT8T', '_score': 0.7587098, '_source': {'text': 'Yosemite National Park is a United States national park covering over 750,000 acres of land in California. A UNESCO World Heritage Site, the park is best known for its granite cliffs, waterfalls, and giant sequoia trees. Yosemite hosts over four million visitors in most years, with a peak of five million visitors in 2016. The park is home to a diverse range of wildlife, including mule deer, black bears, and the endangered Sierra Nevada bighorn sheep. The park has 1,200 square miles of wilderness and is a popular destination for rock climbers. One of its most famous cliffs is El Capitan, a 3,000-foot monolith along its tallest face.'}}, {'_index': 'national-parks', '_id': 'bmxfTp0BAxJsRrxTPT_f', '_score': 0.7587098, '_source': {'text': 'Yosemite National Park is a United States national park covering over 750,000 acres of land in California. A UNESCO World Heritage Site, the park is be

## Load the sample dataset

We download a small fictional workplace dataset and inspect its structure before turning it into LangChain documents.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

with urlopen(url) as response:
    data = json.load(response)

print(f"Loaded {len(data)} source documents.")
print("Example keys:", list(data[0].keys()))

Loaded 15 source documents.
Example keys: ['content', 'summary', 'name', 'url', 'created_on', 'updated_at', 'category', '_run_ml_inference', 'rolePermissions']


## Split documents into passages

We now convert each record into a LangChain `Document`, attach useful metadata, and split the content into overlapping chunks.

We use:

- **chunk size = 800**
- **chunk overlap = 400**

This keeps the lab logic aligned with the original notebook while making retrieval more specific.

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def record_to_document(record: dict) -> Document:
    updated_at_value = record.get("updated_at", "")
    metadata = {
        "name": record.get("name", ""),
        "summary": record.get("summary", ""),
        "url": record.get("url", ""),
        "category": record.get("category", ""),
        "updated_at": updated_at_value if updated_at_value else None, # Set to None if empty string
    }
    return Document(page_content=record["content"], metadata=metadata)

source_docs = [record_to_document(record) for record in data]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=400,
)

docs = text_splitter.split_documents(source_docs)

print(f"Created {len(docs)} passage chunks.")
print(docs[0].metadata)
print(docs[0].page_content[:400])

Created 15 passage chunks.
{'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01'}
Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.
Scope

This policy applies to all employees who are eligible for remote work as determined by their role and responsibilities. It


## Index passages in Elasticsearch

Next, we add the chunks to Elasticsearch so they can be searched by vector similarity.

In [6]:
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_classic.retrievers import MultiQueryRetriever
from elasticsearch import Elasticsearch # Import Elasticsearch client

# Retrieve secrets for initialization
ES_URL = "https://my-elasticsearch-project-ea1419.es.europe-west3.gcp.elastic.cloud:443"
ELASTIC_API_KEY_SECRET = userdata.get(ES_URL).strip()
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()

# Initialize LangChain components
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Define the index name to be used for the LangChain vectorstore
INDEX_NAME = "lab-multi-query-retriever"

# Initialize a direct Elasticsearch client to manage the index
es_client = Elasticsearch(
    hosts=[ES_URL],
    api_key=ELASTIC_API_KEY_SECRET,
    request_timeout=30
)

# Delete the index if it already exists to ensure fresh mapping
if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME, ignore=[400, 404]) # ignore 400 (bad request) or 404 (not found) errors
    print(f"Existing index '{INDEX_NAME}' deleted.")

# Initialize ElasticsearchStore (vectorstore) using es_url and es_api_key
vectorstore = ElasticsearchStore(
    es_url=ES_URL,
    es_api_key=ELASTIC_API_KEY_SECRET,
    index_name=INDEX_NAME,
    embedding=embeddings,
    # Removed es_connection=es_client as it's not a valid argument for ElasticsearchStore
)

vectorstore.add_documents(docs)

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
)

print(f"Indexed {len(docs)} chunks into '{INDEX_NAME}'.")

/tmp/ipykernel_9660/3885363237.py:28: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.delete(index=INDEX_NAME, ignore=[400, 404]) # ignore 400 (bad request) or 404 (not found) errors


Existing index 'lab-multi-query-retriever' deleted.
Indexed 15 chunks into 'lab-multi-query-retriever'.


# Question Answering with MultiQuery Retriever

Now that the passages are stored in Elasticsearch, we can:

1. retrieve relevant context with **MultiQuery Retriever**,
2. combine the retrieved passages,
3. ask the model to answer from that context.

In [8]:
import logging
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, format_document
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Be as educational and clear as possible.

Context:
{context}

Question: {question}
Answer:
"""
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)

def combine_documents(docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)

context_chain = RunnableParallel(
    context=retriever | combine_documents,
    question=RunnablePassthrough(),
)

chain = context_chain | LLM_CONTEXT_PROMPT | llm

question = "What is the NASA sales team?"

answer = chain.invoke(question)

print("---- Question ----")
print(question)
print("\n---- Answer ----")
print(answer.content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What is the role of the sales team at NASA?', "2. How does NASA's sales team contribute to the organization?", '3. Can you provide information about the sales team within NASA?']


---- Question ----
What is the NASA sales team?

---- Answer ----
The NASA sales team is part of the North America South America region within the Americas division of the sales organization. The team is headed by two Area Vice-Presidents: Laura Martinez is the Area Vice-President of North America, and Gary Johnson is the Area Vice-President of South America. The NASA sales team is responsible for serving customers in the United States, Canada, Mexico, Central America, and South America, and achieving business objectives in these regions.


## Reflection

You now have a working MultiQuery Retriever notebook in Colab.

Suggested follow-up exercises:

1. Ask a different workplace question and compare the answer.
2. Change the chunk size or overlap and observe whether retrieval quality changes.
3. Inspect the generated query variations by turning the retriever logger on and asking several questions.

**Generate at least two new iterations of the previous cells. Be creative.**
Did you master MultiQuery Retriever concepts through this lab?

## Iteration 1 — Different Question + Logging the Generated Query Variations

We ask a different workplace question and turn on the MultiQuery logger so we can inspect which alternative queries the retriever generates internally.

In [9]:
import logging

# Enable logging to see the generated query variations
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

question_2 = "What are the work from home policies?"

answer_2 = chain.invoke(question_2)

print("---- Question ----")
print(question_2)
print("\n---- Answer ----")
print(answer_2.content)


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Can you provide information on the remote work guidelines in place?', '2. What are the current policies regarding working remotely?', '3. How does the company handle work from home arrangements?']


---- Question ----
What are the work from home policies?

---- Answer ----
The work from home policies include guidelines and support for employees to work remotely, maintaining productivity during the COVID-19 pandemic and beyond. Eligibility for remote work is determined by the employee's role and responsibilities, requiring approval from the supervisor and HR. Employees receive necessary equipment and resources for remote work, must create a conducive workspace, maintain communication, adhere to regular work hours, meet performance expectations, track time accurately, prioritize confidentiality and data security, and ensure health and well-being. Policies are periodically reviewed and updated, and employees are encouraged to direct any questions or concerns to their supervisor or HR department.


## Iteration 2 — Smaller Chunk Size for More Specific Retrieval

We re-split the documents with a **smaller chunk size (400)** and **less overlap (100)** to see whether more granular chunks improve or change the retrieved context and the final answer.

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_elasticsearch import ElasticsearchStore
from langchain_classic.retrievers import MultiQueryRetriever

# Re-split with smaller chunks
text_splitter_small = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,
    chunk_overlap=100,
)

docs_small = text_splitter_small.split_documents(source_docs)
print(f"Original chunks : {len(docs)}")
print(f"Smaller chunks  : {len(docs_small)}")

# Use a separate index name to avoid mixing chunk sizes
INDEX_NAME_SMALL = "lab-multi-query-retriever-small"

# Clean up the small-chunk index if it already exists
if es_client.indices.exists(index=INDEX_NAME_SMALL):
    es_client.indices.delete(index=INDEX_NAME_SMALL, ignore=[400, 404])
    print(f"Existing index '{INDEX_NAME_SMALL}' deleted.")

vectorstore_small = ElasticsearchStore(
    es_url=ES_URL,
    es_api_key=ELASTIC_API_KEY_SECRET,
    index_name=INDEX_NAME_SMALL,
    embedding=embeddings,
)

vectorstore_small.add_documents(docs_small)

retriever_small = MultiQueryRetriever.from_llm(
    retriever=vectorstore_small.as_retriever(),
    llm=llm,
)

context_chain_small = RunnableParallel(
    context=retriever_small | combine_documents,
    question=RunnablePassthrough(),
)

chain_small = context_chain_small | LLM_CONTEXT_PROMPT | llm

question_3 = "What is the NASA sales team?"
answer_3 = chain_small.invoke(question_3)

print("---- Question ----")
print(question_3)
print("\n---- Answer (small chunks) ----")
print(answer_3.content)


Original chunks : 15
Smaller chunks  : 28


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Can you provide information about the sales team at NASA?', '2. Who makes up the sales team within NASA?', "3. How does NASA's sales team operate?"]


---- Question ----
What is the NASA sales team?

---- Answer (small chunks) ----
The NASA sales team refers to the North America South America region in the sales organization structure. It consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America. The team is responsible for managing sales operations in the North American and South American regions.
